# 09 — Computational Cost Analysis

Quantifies the speedup gained by using surrogates instead of direct simulation.

**Source:** `outputs/phase5/summary.txt`, `outputs/phase0_timing.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()

## Simulator Baseline

In [ ]:
timing = pd.read_csv(ROOT / 'outputs' / 'phase0_timing.csv')
t_analytical = timing['wall_time_s'].mean()
print(f'Analytical simulator: {t_analytical*1000:.4f} ms/eval = {1/t_analytical:.0f} evals/sec')
print(f'ODE simulator (estimated): ~1 s/eval')

## Surrogate Training Cost

In [ ]:
mc1 = pd.read_csv(ROOT / 'outputs' / 'phase3' / 'mc1_summary.csv')
gp_m52 = mc1[(mc1['config_id'] == 'GP-M52') & (mc1['output'] == 'v_out_mean')]

print(f'GP-M52 training (N=100): {gp_m52["train_s"].values[0]:.3f} s')
print(f'GP-M52 prediction: {gp_m52["pred_us"].values[0]:.1f} us/point')

## Speedup Comparison

In [ ]:
tasks = [
    ('Sobol sensitivity (100k evals)', 100_000, 11.11, 27.8 * 3600),
    ('4D design sweep (6.25M evals)', 6_250_000, 830.58, 72.3 * 86400),
    ('2D contour (40k evals)', 40_000, 1.0, 11.1 * 3600),
]

print(f'{"Task":>35s} | {"N evals":>10s} | {"Surrogate":>12s} | {"Direct sim":>15s} | {"Speedup":>10s}')
print('-' * 95)
for name, n, t_surr, t_direct in tasks:
    speedup = t_direct / t_surr
    def fmt_time(s):
        if s < 60: return f'{s:.1f} s'
        if s < 3600: return f'{s/60:.1f} min'
        if s < 86400: return f'{s/3600:.1f} hours'
        return f'{s/86400:.1f} days'
    print(f'{name:>35s} | {n:>10,d} | {fmt_time(t_surr):>12s} | {fmt_time(t_direct):>15s} | {speedup:>9,.0f}x')

## Training Budget Context

In [ ]:
n_train = 187  # CCM-valid training points
t_ode_train = n_train * 1.0  # 1 s/sim for ODE
t_analytical_train = n_train * t_analytical

print(f'Training data generation:')
print(f'  Analytical: {t_analytical_train:.3f} s ({n_train} sims)')
print(f'  ODE baseline: {t_ode_train:.0f} s = {t_ode_train/60:.1f} min')
print()
print(f'Total pipeline cost (training + Sobol):')
print(f'  Via surrogate: {t_ode_train + 11.11:.0f} s = {(t_ode_train + 11.11)/60:.1f} min')
print(f'  Via direct ODE: {100_000:.0f} s = {100_000/3600:.1f} hours')
print(f'  Savings: {100_000 / (t_ode_train + 11.11):.0f}x')

## Conclusion

The surrogate-based pipeline achieves **7,500-9,000x speedup** for downstream analysis tasks.
Even including the training data generation cost (~3 min for 187 ODE simulations),
the total pipeline is orders of magnitude faster than direct simulation for tasks
requiring >1,000 evaluations.